# 06 - Graph Inspection and Visualization

Orthograph provides a two-phase architecture for understanding graph data:

1. **Inspect** -- a `GraphInspector` scans a graph source (NetworkX, Neo4j, etc.) and produces a `GraphProfile`: a structural summary of node types, relationship types, property completeness, observed types, and cardinality statistics.
2. **Validate** -- the `validate_profile` function compares a `GraphProfile` against a `GraphDataModel` and reports mismatches (missing labels, type conflicts, incomplete properties, unexpected entities).

This notebook demonstrates the full workflow using NetworkX as the graph backend:

- Building a NetworkX graph with sample data
- Inspecting it with `NetworkxInspector` to get a `GraphProfile`
- Exploring the profile (node/relationship counts, property completeness, cardinality)
- Validating the profile against the model with `validate_profile`
- Visualizing the model schema with `schema_to_networkx` and `to_mermaid`

In [1]:
from typing import Optional

import networkx as nx

from orthograph import (
    Cardinality,
    GraphDataModel,
    NodeModel,
    RelationshipModel,
)
from orthograph.extensions.networkx import NetworkxInspector, schema_to_networkx
from orthograph.extensions.validation import validate_profile
from orthograph.extensions.visualization import to_mermaid

## Define the model

We use a chemistry domain: molecules participate as reactants or products in chemical equations, and each equation can have a reaction template.

In [2]:
class Molecule(NodeModel):
    __label__ = "Molecule"
    __uid_field__ = "uid"
    uid: str
    smiles: str


class ChemicalEquation(NodeModel):
    __label__ = "ChemicalEquation"
    __uid_field__ = "uid"
    uid: str
    smiles: str


class Template(NodeModel):
    __label__ = "Template"
    __uid_field__ = "uid"
    uid: str
    smarts: str


class Reactant(RelationshipModel):
    __label__ = "REACTANT"
    __source_type__ = Molecule
    __target_type__ = ChemicalEquation
    __source_cardinality__ = Cardinality.ZERO_OR_MORE
    __target_cardinality__ = Cardinality.ONE_OR_MORE


class Product(RelationshipModel):
    __label__ = "PRODUCT"
    __source_type__ = ChemicalEquation
    __target_type__ = Molecule
    __source_cardinality__ = Cardinality.ONE_OR_MORE
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


class HasTemplate(RelationshipModel):
    __label__ = "HAS_TEMPLATE"
    __source_type__ = ChemicalEquation
    __target_type__ = Template
    __source_cardinality__ = Cardinality.ZERO_OR_ONE


model = GraphDataModel(
    name="Chemistry",
    node_types=[Molecule, ChemicalEquation, Template],
    relationship_types=[Reactant, Product, HasTemplate],
)

print("Model:", model.name)
print("Nodes:", model.node_labels)
print("Rels: ", model.relationship_labels)

Model: Chemistry
Nodes: {'Template', 'Molecule', 'ChemicalEquation'}
Rels:  {'REACTANT', 'HAS_TEMPLATE', 'PRODUCT'}


## Build a NetworkX graph with sample data

We populate a `MultiDiGraph` with molecules, equations, and templates. To demonstrate property completeness analysis, we intentionally omit the `smiles` property on one Molecule node.

In [3]:
G = nx.MultiDiGraph()

# Molecules -- note mol_3 is missing the 'smiles' property
G.add_node("mol_1", __label__="Molecule", uid="mol_1", smiles="CCO")
G.add_node("mol_2", __label__="Molecule", uid="mol_2", smiles="CC=O")
G.add_node("mol_3", __label__="Molecule", uid="mol_3")  # missing 'smiles'

# Chemical equations
G.add_node("ce_1", __label__="ChemicalEquation", uid="ce_1", smiles="CCO>>CC=O")
G.add_node("ce_2", __label__="ChemicalEquation", uid="ce_2", smiles="CC=O>>CCC")

# Templates
G.add_node("tpl_1", __label__="Template", uid="tpl_1", smarts="[C:1][OH]>>[C:1]=O")

# Relationships
G.add_edge("mol_1", "ce_1", __label__="REACTANT")
G.add_edge("mol_3", "ce_2", __label__="REACTANT")
G.add_edge("ce_1", "mol_2", __label__="PRODUCT")
G.add_edge("ce_2", "mol_1", __label__="PRODUCT")
G.add_edge("ce_1", "tpl_1", __label__="HAS_TEMPLATE")

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

Graph has 6 nodes and 5 edges


## Inspect the graph

`NetworkxInspector` scans the graph and produces a `GraphProfile` -- a frozen Pydantic model that summarizes everything about the graph's structure.

In [5]:
inspector = NetworkxInspector(G)
profile = inspector.inspect()

print("Profile source:   ", profile.source)
print("Profile timestamp:", profile.timestamp)
print("Node labels:      ", profile.node_labels)
print("Relationship types:", profile.relationship_types)

Profile source:    networkx
Profile timestamp: 2026-04-14 12:02:01.186501
Node labels:       {'Template', 'Molecule', 'ChemicalEquation'}
Relationship types: {'REACTANT', 'HAS_TEMPLATE', 'PRODUCT'}


## Explore node type profiles

Each `NodeTypeProfile` contains the label, instance count, and a `PropertyProfile` for every property observed across all instances of that type. The property profile tracks completeness (what fraction of nodes have the property) and observed Python types.

In [6]:
for label, ntp in profile.node_type_profiles.items():
    print(f"--- {label} ({ntp.count} instances) ---")
    for prop_name, pp in ntp.property_profiles.items():
        print(
            f"  {prop_name:12s}  "
            f"completeness={pp.completeness:.0%}  "
            f"({pp.present_count}/{pp.total_count})  "
            f"types={pp.observed_types}"
        )
    print()

--- ChemicalEquation (2 instances) ---
  smiles        completeness=100%  (2/2)  types=['str']
  uid           completeness=100%  (2/2)  types=['str']

--- Molecule (3 instances) ---
  smiles        completeness=67%  (2/3)  types=['str']
  uid           completeness=100%  (3/3)  types=['str']

--- Template (1 instances) ---
  smarts        completeness=100%  (1/1)  types=['str']
  uid           completeness=100%  (1/1)  types=['str']



Notice that the `smiles` property on `Molecule` has only 67% completeness -- `mol_3` is missing it. The `is_mandatory` flag on that property profile will be `False`.

## Explore relationship type profiles

Each `RelationshipTypeProfile` contains the relationship type, count, source/target labels, property profiles, and cardinality statistics (min/max/avg outgoing degree from source nodes).

In [7]:
for rel_type, rtp in profile.rel_type_profiles.items():
    print(f"--- {rel_type} ({rtp.count} instances) ---")
    print(f"  source labels: {rtp.source_labels}")
    print(f"  target labels: {rtp.target_labels}")
    if rtp.cardinality_stats:
        cs = rtp.cardinality_stats
        print(
            f"  cardinality:   min={cs.min_degree}, max={cs.max_degree}, "
            f"avg={cs.avg_degree:.1f}, sample_size={cs.sample_size}"
        )
    print()

--- HAS_TEMPLATE (1 instances) ---
  source labels: {'ChemicalEquation'}
  target labels: {'Template'}
  cardinality:   min=1, max=1, avg=1.0, sample_size=1

--- PRODUCT (2 instances) ---
  source labels: {'ChemicalEquation'}
  target labels: {'Molecule'}
  cardinality:   min=1, max=1, avg=1.0, sample_size=2

--- REACTANT (2 instances) ---
  source labels: {'Molecule'}
  target labels: {'ChemicalEquation'}
  cardinality:   min=1, max=1, avg=1.0, sample_size=2



## Validate the profile against the model

`validate_profile` compares the `GraphProfile` against the `GraphDataModel` and reports:

- **Errors**: missing labels, type mismatches, invalid endpoints, cardinality violations
- **Warnings**: incomplete required properties, unexpected labels
- **Info**: unexpected properties not in the model

In [8]:
result = validate_profile(profile, model)

print("is_valid:", result.is_valid)
print(f"Errors:   {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")
print()

for issue in result.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.message}")
    if issue.context:
        print(f"    context: {issue.context}")

is_valid: True
Errors:   0
Warnings: 1

  [warning] [PROPERTY_INCOMPLETE] Required property 'smiles' on Molecule is only 66.7% complete
    context: {'present_count': 2, 'total_count': 3, 'completeness': 0.6666666666666666}


The validation caught that `smiles` on `Molecule` is a required property in the model but is only 67% complete in the data. Since this is a completeness issue (not a total absence), it is reported as a **warning** rather than an error.

## Visualize the model schema as a NetworkX graph

The `schema_to_networkx` function converts the model definition itself -- not instance data -- into a NetworkX `MultiDiGraph`. Nodes in this graph represent node types; edges represent relationship types. This is useful for programmatic analysis of the schema structure.

In [9]:
schema_graph = schema_to_networkx(model)

print("Schema graph nodes:")
for node, attrs in schema_graph.nodes(data=True):
    print(f"  {node}: uid_field={attrs['uid_field']}, properties={attrs['properties']}")

print()
print("Schema graph edges:")
for src, tgt, attrs in schema_graph.edges(data=True):
    print(f"  {src} --[{attrs['label']}]--> {tgt}")
    print(f"    source_cardinality={attrs['source_cardinality']}, target_cardinality={attrs['target_cardinality']}")

Schema graph nodes:
  Molecule: uid_field=uid, properties={'uid': 'str', 'smiles': 'str'}
  ChemicalEquation: uid_field=uid, properties={'uid': 'str', 'smiles': 'str'}
  Template: uid_field=uid, properties={'uid': 'str', 'smarts': 'str'}

Schema graph edges:
  Molecule --[REACTANT]--> ChemicalEquation
    source_cardinality=min=0 max=None, target_cardinality=min=1 max=None
  ChemicalEquation --[PRODUCT]--> Molecule
    source_cardinality=min=1 max=None, target_cardinality=min=0 max=None
  ChemicalEquation --[HAS_TEMPLATE]--> Template
    source_cardinality=min=0 max=1, target_cardinality=min=0 max=None


## Generate a Mermaid diagram

The `to_mermaid` function generates a text-based Mermaid diagram of the model schema. This can be rendered in any Mermaid-compatible viewer (GitHub markdown, mermaid.live, Jupyter with mermaid extensions, etc.).

In [10]:
mermaid_text = to_mermaid(model)
print(mermaid_text)

graph TD
    Molecule["Molecule<br>uid: str, smiles: str"]
    ChemicalEquation["ChemicalEquation<br>uid: str, smiles: str"]
    Template["Template<br>uid: str, smarts: str"]
    Molecule -->|REACTANT| ChemicalEquation
    ChemicalEquation -->|PRODUCT| Molecule
    ChemicalEquation -->|HAS_TEMPLATE| Template


The Mermaid diagram below will render in viewers that support `mermaid` fenced code blocks (GitHub, mermaid.live, Jupyter with appropriate extensions).

```mermaid
graph TD
    Molecule["Molecule<br>uid: str, smiles: str"]
    ChemicalEquation["ChemicalEquation<br>uid: str, smiles: str"]
    Template["Template<br>uid: str, smarts: str"]
    Molecule -->|REACTANT| ChemicalEquation
    ChemicalEquation -->|PRODUCT| Molecule
    ChemicalEquation -->|HAS_TEMPLATE| Template
```